In [18]:
import numpy as np

def evaluate_traffic_delay(green_times):
    g1, g2 = green_times
    if g1 < 10 or g1 > 120 or g2 < 10 or g2 > 120:
        return 1e6

    yellow_time = 4.0
    cycle_length = g1 + g2 + (2 * yellow_time)

    # Red light times are determined by the opposing phase's duration
    r1 = g2 + (2 * yellow_time)
    r2 = g1 + (2 * yellow_time)

    # --- UNBALANCED TRAFFIC DEMAND ---
    arrival_rate1 = 0.65  # Heavy rush hour traffic on the main road (Phase 1)
    arrival_rate2 = 0.20  # Light traffic on the cross street (Phase 2)
    saturation_flow = 1.0 # Max discharge capacity (1 car/sec)

    # Calculate how many cars pile up during the red light
    cars_waiting_phase1 = r1 * arrival_rate1
    cars_waiting_phase2 = r2 * arrival_rate2

    # Maximum number of cars that can physically clear during the green light
    max_clearing_capacity1 = g1 * saturation_flow
    max_clearing_capacity2 = g2 * saturation_flow

    # GRIDLOCK PENALTY: If the green light is too short to clear the queue,
    # add a massive penalty so the PSO rejects this timing configuration.
    if cars_waiting_phase1 > max_clearing_capacity1 or cars_waiting_phase2 > max_clearing_capacity2:
        return 1e5 + (cars_waiting_phase1 - max_clearing_capacity1) * 500

    # Calculate uniform queueing delays
    delay1 = 0.5 * cars_waiting_phase1 * r1
    delay2 = 0.5 * cars_waiting_phase2 * r2

    return delay1 + delay2

# PSO Parameters
num_particles = 15
max_iterations = 30
w = 0.5   # Inertia weight
c1 = 1.5  # Cognitive weight
c2 = 1.5  # Social weight

# Search space bounds
lb = np.array([10.0, 10.0])
ub = np.array([120.0, 120.0])
dim = 2

# Initialize particle positions and velocities
positions = np.random.uniform(lb, ub, (num_particles, dim))
velocities = np.zeros((num_particles, dim))

# Initialize personal bests
pbest_positions = np.copy(positions)
pbest_scores = np.array([evaluate_traffic_delay(p) for p in positions])

# Initialize global best
gbest_idx = np.argmin(pbest_scores)
gbest_position = np.copy(pbest_positions[gbest_idx])
gbest_score = pbest_scores[gbest_idx]

# Main PSO Loop
for iteration in range(max_iterations):
    for i in range(num_particles):
        r1_rand, r2_rand = np.random.rand(2)

        # Update velocity
        velocities[i] = (w * velocities[i] +
                         c1 * r1_rand * (pbest_positions[i] - positions[i]) +
                         c2 * r2_rand * (gbest_position - positions[i]))

        # Update position
        positions[i] += velocities[i]

        # Clamp positions to search space bounds
        positions[i] = np.clip(positions[i], lb, ub)

        # Evaluate new position
        score = evaluate_traffic_delay(positions[i])

        # Update personal best
        if score < pbest_scores[i]:
            pbest_scores[i] = score
            pbest_positions[i] = np.copy(positions[i])

        # Update global best
        if score < gbest_score:
            gbest_score = score
            gbest_position = np.copy(positions[i])


print("\n--- Optimization Complete ---")
print(f"Optimal Green Light Times (Phase 1, Phase 2): [{gbest_position[0]:.2f}s, {gbest_position[1]:.2f}s]")
print(f"Minimum Estimated Total Delay: {gbest_score:.2f} car-seconds per cycle")



--- Optimization Complete ---
Optimal Green Light Times (Phase 1, Phase 2): [11.70s, 10.00s]
Minimum Estimated Total Delay: 144.11 car-seconds per cycle
